# ICLR Inference on Real Robot Data

## Setup Model for Inference

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

%load_ext autoreload
%autoreload 2
import h5py
import numpy as np
from iclr.models.policy_real.iclr_real_wrapper import ICLRRealWrapper
from PIL import Image
import pickle
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from tqdm import trange
from IPython.display import HTML

matplotlib.rcParams['animation.embed_limit'] = 100.0

In [2]:
# More configuration for checkpoint path, vision encoder weights, etc.
# Your train pth file
checkpoint_path = "/data/tientoan/iclr_output/output_real/iclr_real_0/checkpoint-102.pth"

# Remember to copy run.yaml in your checkpoint folder to an inference.yaml file. In inference.yaml,
# change scratch_llama_config: config/model_config/custom_transformer.json to
# scratch_llama_config: ../config/model_config/custom_transformer.json
train_yaml_path = "/data/tientoan/iclr_output/output_real/iclr_real_0/inference.yaml"

# pretrained vision encoder pth file
vision_encoder_path = "../vision_encoder/cross-mae-rtx-vitb.pth"

In [ ]:
iclr_real_wrapper = ICLRRealWrapper(train_yaml_path, checkpoint_path, vision_encoder_path)

## Data Visualization Example

In [4]:
def draw_points_on_image(image, points, radius=5, line_width=5):
    """
    Draw num_visual_trace_points 2D points on a PIL Image, and lines between consecutive points with increased width,
    using unique colors per point. The color of each line is the same as the start-point.
    For 5 points: '990000', 'CC0000', 'FF0000', 'FF3333', 'FF3333'
    """
    # Define the color codes as RGB tuples
    hex_colors = ['990000', 'CC0000', 'FF0000', 'FF3333', 'FF3333']
    color_tuples = [tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4)) for hex_color in hex_colors]

    img = image.copy()
    np_img = np.array(img)
    h, w = np_img.shape[:2]

    # Draw the points, each with its unique color
    yy, xx = np.ogrid[:h, :w]
    for idx, (u, v) in enumerate(points):
        color = color_tuples[min(idx, len(color_tuples)-1)]
        mask = (xx - u) ** 2 + (yy - v) ** 2 <= radius ** 2
        np_img[mask] = color

    # Draw lines between consecutive points using color of starting point, with increased thickness
    def draw_thick_pixel(np_img, x, y, color, width):
        for dx in range(-width, width + 1):
            for dy in range(-width, width + 1):
                if dx*dx + dy*dy <= width*width:
                    xx, yy = x + dx, y + dy
                    if 0 <= xx < np_img.shape[1] and 0 <= yy < np_img.shape[0]:
                        np_img[yy, xx] = color

    for i in range(len(points) - 1):
        u0, v0 = points[i]
        u1, v1 = points[i+1]
        color = color_tuples[min(i, len(color_tuples)-1)]
        num_interp = int(np.hypot(u1-u0, v1-v0)) * 2  # Simple dense enough sampling
        for t in np.linspace(0, 1, max(2, num_interp)):
            x = int(round(u0 * (1-t) + u1 * t))
            y = int(round(v0 * (1-t) + v1 * t))
            if 0 <= x < w and 0 <= y < h:
                draw_thick_pixel(np_img, x, y, color, line_width)

    return Image.fromarray(np_img)

In [5]:
def make_visual_trace(points, n=5):
    """
    Given (N, 2) points, return (N, n, 2) where for each point i,
    we take n evenly spaced indices between i and N-1 (inclusive),
    and collect those points from the original array.
    """
    N = len(points)
    result = np.zeros((N, n, 2), dtype=points.dtype)

    for i in range(N):
        idxs = np.linspace(i, N - 1, n, dtype=int)
        result[i] = points[idxs]
    return result

In [6]:
def get_data_from_h5(data, visual_trace_data, episode_name, resolution=(240, 424, 3),
                return_PIL_images=False, num_visual_trace_points=5):
    # img, action and proprio keys can be found in config/dataset_config_template.yaml
    side_images = np.frombuffer(data[f"{episode_name}/observation/exterior_image_1_left"][:], dtype="uint8").reshape(-1, *resolution)
    wrist_images = np.frombuffer(data[f"{episode_name}/observation/wrist_image_left"][:], dtype="uint8").reshape(-1, *resolution)

    action_keys = ["action/cartesian_position", "action/gripper_position"]
    proprio_keys = ["observation/cartesian_position", "observation/gripper_position"] 
    actions = np.concatenate([data[f"{episode_name}/{key}"][:] for key in action_keys], axis=-1)
    proprios = np.concatenate([data[f"{episode_name}/{key}"][:] for key in proprio_keys], axis=-1)

    if return_PIL_images:
        side_images = [Image.fromarray(side_images[i]) for i in range(side_images.shape[0])]
        wrist_images = [Image.fromarray(wrist_images[i]) for i in range(wrist_images.shape[0])]

    trace = np.array(visual_trace_data[episode_name]) / np.array([1000., 1000.])
    trace = make_visual_trace(trace, n=num_visual_trace_points)
    visual_trace = trace.reshape(-1, num_visual_trace_points * 2)   # (T, 10)


    return {
        "side_images": side_images,
        "wrist_images": wrist_images,
        "visual_trace": visual_trace,
        "actions": actions,
        "proprios": proprios,
    }

In [7]:
# Visual trace pickle file

# Change the path below to your path to the real visual trace data
visual_trace_path = "/data/tientoan/ICL_Franka/visual_trace_real.pkl"
with open(visual_trace_path, "rb") as f:
    # load visual trace data
    visual_trace_data = pickle.load(f)

In [ ]:
## Visualize the Dataset 
dataset_path = "/data/tientoan/ICL_Franka/iclr_real_data.hdf5"   # change to your data path
data = h5py.File(dataset_path, "r")

resolution = (240, 424, 3)

# Load the dataset
# Pick an episode name
episode_name = "dumpling_blue_bowl_10"
print("Selected demo for visualization: ", episode_name) 

obs_dict = get_data_from_h5(data, visual_trace_data, episode_name, resolution)
side_images, wrist_images, visual_trace, proprios, actions = obs_dict["side_images"], obs_dict["wrist_images"], obs_dict["visual_trace"], obs_dict["proprios"], obs_dict["actions"]

In [ ]:
# draw visual traces on the third-view camera images
h, w = resolution[0], resolution[1]
traced_side_images = [
    draw_points_on_image(side_image, visual_trace[i].reshape(-1, 2) * np.array([w, h]), line_width=1)
    for i, side_image in enumerate(side_images)
]
frames = np.concatenate([traced_side_images, wrist_images], axis=1)
T = frames.shape[0]

fig, ax = plt.subplots()
img = ax.imshow(frames[0])

def update(frame):
    img.set_data(frame)
    return [img]

ani = animation.FuncAnimation(fig, update, frames=frames, interval=50, blit=True)

In [ ]:
# Run the visualization
HTML(ani.to_jshtml())

## Inference with Teacher Forcing

Since this code is not using the real robot, we are using teacher forcing here just for the verification purpose. When running on the real robot, remember to disable the teacher forcing.

### Prompting

In [ ]:
# Select a prompt demo
episode_name = "jaguar_red_box_25"
print("Selected prompt demo: ", episode_name)

obs_dict = get_data_from_h5(data, visual_trace_data, episode_name, resolution, return_PIL_images=True)
side_images, wrist_images, visual_trace, proprios, actions = (
    obs_dict["side_images"], obs_dict["wrist_images"], obs_dict["visual_trace"],
    obs_dict["proprios"], obs_dict["actions"],
)

iclr_real_wrapper.reset()
iclr_real_wrapper.prompt(
    side_images,
    wrist_images,
    proprios,
    visual_trace,
    actions,
)


In [ ]:
# Select an episode for testing with teacher forcing
episode_name = "jaguar_red_box_40"
print("Selected episode for testing with teacher forcing: ", episode_name)

obs_dict = get_data_from_h5(data, visual_trace_data, episode_name, resolution,
                            return_PIL_images=True)          # <-- PIL, not numpy
side_images, wrist_images, visual_trace, proprios, actions = (
    obs_dict["side_images"], obs_dict["wrist_images"], obs_dict["visual_trace"],
    obs_dict["proprios"], obs_dict["actions"],
)

pred_actions, pred_traces = [], []
for i in trange(len(side_images)):
    action = None if i == 0 else actions[i-1:i]
    new_action, new_trace = iclr_real_wrapper(
        side_images[i],
        wrist_images[i],
        proprios[i:i+1],
        visual_trace[i:i+1],
        action=action,
        use_temporal=False,
        teacher_forcing=True,
    )
    pred_actions.append(new_action)
    pred_traces.append(new_trace)



In [ ]:
# plot the predicted actions and ground truth actions for each axis, (x,y,z,r,p,y,gripper)
pred_actions = np.asarray(pred_actions)
action_keys = ["x", "y", "z", "roll", "pitch", "yaw", "gripper"]

T = len(pred_actions)
plt.figure(figsize=(14, 6))
for i in range(7):
    plt.subplot(2, 4, i+1)
    plt.plot(range(T), pred_actions[:, i], label='Predicted')
    plt.plot(range(T), actions[:T, i], label='Ground Truth')
    plt.xlabel('Time Step')
    plt.ylabel('Action Value')
    plt.title(action_keys[i])
    plt.legend()

plt.tight_layout()
plt.show()

For more information, please refer to `iclr/models/policy_real/iclr_real_wrapper.py`